# 🎬 BigByte Systems: Free Cloud GPU Talking Avatar Video Generator
Generate 100% realistic, lip-synced studio podcast interview videos for **Host Ananya** and **Specialist Vikram**.

### ⚡ Zero Cost & Zero Load on Your PC
- **Hardware**: Google Cloud 16GB NVIDIA Tesla T4 GPU (₹0 cost).
- **Execution Time**: ~2 to 3 minutes.
- **Instruction**: Click **Runtime** ➔ **Run all** (or press `Ctrl + F9`). Both videos will be bundled into `podcast_videos.zip` and downloaded to your browser!

In [ ]:
# 1. Clone SadTalker & Install Bulletproof Compatible Dependencies
!git clone https://github.com/OpenTalker/SadTalker.git
%cd /content/SadTalker

# Pin compatible versions to prevent NumPy 2.x and torchvision breaking changes
!pip install -q "numpy==1.23.5" "numba<0.57"
!pip install -q "torchvision>=0.13" "torchaudio"
!pip install -q -r requirements.txt
!pip install -q imageio-ffmpeg facexlib kornia yacs gfpgan
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2

# CRUCIAL FIX: Patch BasicSR for modern torchvision (prevents functional_tensor ModuleNotFoundError)
!find /usr/local/lib/python* -name degradations.py -exec sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' {} +

print('✅ Dependencies installed & torchvision compatibility patch applied!')

In [ ]:
# 2. Download Pre-Trained Neural Weights & 3D Basel Face Model
%cd /content/SadTalker
!mkdir -p ./checkpoints/BFM_Fitting ./gfpgan/weights

# Main SadTalker Safetensors & Mapping
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar -d ./checkpoints -o mapping_00109-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar -d ./checkpoints -o mapping_00229-model.pth.tar
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors -d ./checkpoints -o SadTalker_V0.0.2_256.safetensors

# 3D Morphable Model (BFM_Fitting)
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/BFM_Fitting.zip -d ./checkpoints -o BFM_Fitting.zip
!unzip -o -q ./checkpoints/BFM_Fitting.zip -d ./checkpoints/BFM_Fitting/
!if [ -d "./checkpoints/BFM_Fitting/BFM_Fitting" ]; then mv ./checkpoints/BFM_Fitting/BFM_Fitting/* ./checkpoints/BFM_Fitting/ && rm -rf ./checkpoints/BFM_Fitting/BFM_Fitting; fi

# PyTorch Hub Models
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/hub.zip -d ./checkpoints -o hub.zip
!unzip -o -q ./checkpoints/hub.zip -d ./checkpoints/

# GFPGAN Face Enhancer Weights
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth -d ./gfpgan/weights -o alignment_WFLW_4HG.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth -d ./gfpgan/weights -o detection_Resnet50_Final.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -d ./gfpgan/weights -o GFPGANv1.4.pth
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth -d ./gfpgan/weights -o parsing_parsenet.pth

print('✅ All neural checkpoints and 3D face models verified!')

In [ ]:
# 3. Fetch Portraits & Neural Speech Audio from your GitHub repo
%cd /content/SadTalker

!wget -q -O ananya.jpg https://raw.githubusercontent.com/bbsthane-web/bigbyte-hyperlocal/master/assets/images/interviewer_ananya.jpg
!wget -q -O vikram.jpg https://raw.githubusercontent.com/bbsthane-web/bigbyte-hyperlocal/master/assets/images/expert_vikram.jpg
!wget -q -O audio_ananya.mp3 https://raw.githubusercontent.com/bbsthane-web/bigbyte-hyperlocal/master/assets/audio/interview_vsl/seg1_ananya.mp3
!wget -q -O audio_vikram.mp3 https://raw.githubusercontent.com/bbsthane-web/bigbyte-hyperlocal/master/assets/audio/interview_vsl/seg1_vikram.mp3

# Convert to 16kHz mono WAV for high-fidelity lip-sync extraction
!ffmpeg -y -i audio_ananya.mp3 -ar 16000 -ac 1 audio_ananya.wav
!ffmpeg -y -i audio_vikram.mp3 -ar 16000 -ac 1 audio_vikram.wav

# 4. Render Lip-Synced Video for Ananya (Host)
print('🎬 [1/2] Generating Ananya Talking Video...')
!python inference.py \
    --driven_audio audio_ananya.wav \
    --source_image ananya.jpg \
    --result_dir ./results_ananya \
    --still \
    --preprocess crop \
    --enhancer gfpgan

# 5. Render Lip-Synced Video for Vikram (Growth Architect)
print('🎬 [2/2] Generating Vikram Talking Video...')
!python inference.py \
    --driven_audio audio_vikram.wav \
    --source_image vikram.jpg \
    --result_dir ./results_vikram \
    --still \
    --preprocess crop \
    --enhancer gfpgan

In [ ]:
# 6. Package Both Videos into a Single ZIP & Download
import glob, shutil, os, zipfile
from google.colab import files

jobs = [
    ('ananya_podcast.mp4', './results_ananya'),
    ('vikram_podcast.mp4', './results_vikram')
]

found = []
for filename, folder in jobs:
    videos = glob.glob(f'{folder}/**/*.mp4', recursive=True)
    if videos:
        latest = max(videos, key=os.path.getmtime)
        shutil.copy(latest, filename)
        found.append(filename)
        print(f'✅ Found video: {latest} -> {filename} ({os.path.getsize(filename):,} bytes)')
    else:
        print(f'⚠️ No video found in {folder}')

if found:
    zip_name = 'podcast_videos.zip'
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in found:
            zf.write(f, arcname=f)
    print(f'\n📦 Created single bundle: {zip_name} ({os.path.getsize(zip_name):,} bytes)')
    print('🚀 Downloading podcast_videos.zip to your Downloads folder...')
    files.download(zip_name)
    print('\n👉 Once downloaded, run Sync-Podcast-Videos.bat on your computer!')
else:
    print('❌ No videos found to package. Please check the inference logs above for errors.')

In [ ]:
# 7. Watch Generated Videos Live in Colab!
from IPython.display import HTML, display
from base64 import b64encode
import os

for filename in ['ananya_podcast.mp4', 'vikram_podcast.mp4']:
    if os.path.exists(filename):
        data_url = 'data:video/mp4;base64,' + b64encode(open(filename, 'rb').read()).decode()
        display(HTML(f'''
        <div style="margin: 15px 0;">
            <h4 style="font-family: sans-serif; color: #10b981;">🎬 {filename}</h4>
            <video width="320" height="320" controls autoplay loop style="border-radius: 12px; box-shadow: 0 4px 20px rgba(0,0,0,0.3);">
                <source src="{data_url}" type="video/mp4">
            </video>
        </div>
        '''))